# 04 · Stellar locus

Overlays the Rubin COSMOS stellar locus (median star colours against g−i, with robust widths) as red ±2σ bands on the
four adjacent-band colour–colour diagrams, and measures how far stars with Gaia parallaxes sit from it.

| Code | Figure |
|---|---|
| L1 | all DIA objects |
| L2 | stars with Gaia parallax S/N > 5 |
| L3 | by SIMBAD group |
| L4 | by Fink label |
| L5 | the g−i > 2 objects |
| L6 | the g−i > 2 objects by Fink label |

Galactic bulge and plane fields are excluded, because the colours carry no extinction correction.

**Inputs:** notebooks 01 and 02, plus the locus file (`LOCUS_FILE` in `_setup.py`). The file's column header is a
`#` comment line; the kit's reader handles it.
**Outputs:** `RUN_ROOT/04_stellar_locus/`, holding the figures, the exported table `fink_dia_template_colours.csv.gz`
(the same table the laptop plot kit uses) and `tables/gaia_star_locus_residuals.json`. Run time: about 1 minute.

In [ ]:
# Shared configuration: paths, run folder, helpers (edit notebooks/science/_setup.py, not this cell).
from _setup import *
describe_config()
import export_colour_plot_table as export
import plot_colour_diagrams as kit
import fink_checks
template_photometry = require(RUN_ROOT / "01_template_colours/tables/dia_object_template_photometry.parquet", "01_template_colours")
science_table = require(RUN_ROOT / "02_science_suite/tables/dia_object_science_table.parquet", "02_science_suite")
locus = require(LOCUS_FILE, "the locus file (see LOCUS_FILE in _setup.py)")
out = step_dir("04_stellar_locus")

## Step 1 · Export the per-object colour table
One row per DIA object with usable template magnitudes in at least two bands.

In [ ]:
table = export.build_table(template_photometry, science_table)
table_path = export.write_table(table, out)
print(f"{len(table):,} objects ->", table_path)

## Step 2 · Read the stellar locus

In [ ]:
L = kit.load_locus(locus)
print(len(L), "rows; g-i from", L["g-i"].min(), "to", L["g-i"].max())
L.head()

## Step 3 · Figures L1–L6

In [ ]:
d = kit.load_data(table_path)
figures = [
    kit.make_figure(d, locus, out / "figures", name="L1_all_objects"),
    kit.make_figure(d, locus, out / "figures", name="L2_gaia_parallax_stars", select="gaia_parallax_mas / gaia_parallax_err_mas > 5"),
    kit.make_figure(d, locus, out / "figures", name="L3_by_simbad_group", facet="simbad_group"),
    kit.make_figure(d, locus, out / "figures", name="L4_by_fink_label", facet="fink_cats_label"),
    kit.make_figure(d, locus, out / "figures", name="L5_gi_gt2", select="has_gri and gi > 2"),
    kit.make_figure(d, locus, out / "figures", name="L6_gi_gt2_by_fink_label", select="has_gri and gi > 2", facet="fink_cats_label"),
]
show(figures)

## Step 4 · Do our stars sit on the locus?
For stars with Gaia parallax S/N > 5: each colour minus the locus colour at the star's own g−i. The g−r and r−i offsets
are equal and opposite by construction, because both are evaluated at fixed g−i.

In [ ]:
resid = fink_checks.gaia_locus_residuals(d, locus)
save_json(resid, out / "tables" / "gaia_star_locus_residuals.json")
import pandas as pd
pd.DataFrame({c: resid[c] for c in ("u-g", "g-r", "r-i", "i-z", "z-y")}).T